In [1]:
import os
import subprocess

import ollama_setup

In [2]:
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    subprocess.run(['apt-get', 'update'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'zstd'], check=True)
    subprocess.run(['pip', 'install', 'ollama'], check=True)

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [81.1 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,147 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,799 kB]
Get:14 http:/

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 54 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (642 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125658 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
ollama_setup.download_and_install()

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...


Ollama has been installed.


>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
ollama_process = ollama_setup.start_background_process()

Ollama is now running in the background.


In [5]:
import ollama
from tqdm.notebook import tqdm

from llm_printing import print_dictionary

# Decision Models

### I modelli decisionali sono LLM che anziché generare testo, compiono decisioni.

### Al contrario degli LLM che producono testo, la rete neurale viene interrogata una sola volta.

### Vengono distinti tre tipi primitivi di decisioni: `noul`, `choice` e `score`.

In [6]:
DECISION_MODEL_NAME = 'nimble-v3:9b-q4_K_M'

In [7]:
ollama.pull(DECISION_MODEL_NAME)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [8]:
TICKET = 'I was charged twice. Refund the extra payment immediately!'

## Noul

 ### Un `noul` è una domanda binaria: la risposta può essere solo `True` o `False`.

In [9]:
REFUND_NOUL_NAME = 'refund'

In [10]:
REFUND_NOUL = {
    REFUND_NOUL_NAME: {
        'type': 'noul',
        'instructions': 'Does the customer explicitly ask for a refund?'
    },
}

In [11]:
print_dictionary(REFUND_NOUL)

{
    "refund": {
        "type": "noul",
        "instructions": "Does the customer explicitly ask for a refund?"
    }
}


In [12]:
response = ollama.systemone(DECISION_MODEL_NAME, TICKET, REFUND_NOUL)

### Facoltativamente, è possibile indicare esplicitamente cosa si intende per `True` e `False`.

In [13]:
REFUND_NOUL = {
    REFUND_NOUL_NAME: {
        'type': 'noul',
        'instructions': 'Does the customer explicitly ask for a refund?',
        'criteria': {
            'true': 'The customer asks to be paid back because they are disattisfied or an error occured.',
            'false': 'The customer may refer to money, but not asking to be paid back.'
        }
    },
}

In [14]:
print_dictionary(REFUND_NOUL)

{
    "refund": {
        "type": "noul",
        "instructions": "Does the customer explicitly ask for a refund?",
        "criteria": {
            "true": "The customer asks to be paid back because they are disattisfied or an error occured.",
            "false": "The customer may refer to money, but not asking to be paid back."
        }
    }
}


In [15]:
response = ollama.systemone(
    model=DECISION_MODEL_NAME,
    state=TICKET,
    questions=REFUND_NOUL
)

### L'output è un punteggio tra `0.0` e `1.0`, dove `0.0` vuol dire `False`, mentre `1.0` vuol dire `True`.

### Può essere saggio considerare un valore troppo vicino a `0.5` come una risposta "incerta", della quale è meglio non fidarsi troppo.

In [16]:
POSITIVE_NOUL_THRESHOLD = 0.6
NEGATIVE_NOUL_THRESHOLD = 0.4

In [17]:
noul = response.answers[REFUND_NOUL_NAME].noul

if noul > POSITIVE_NOUL_THRESHOLD:
    print('Refunding request detected. Sigh...')
elif noul < NEGATIVE_NOUL_THRESHOLD:
    print('Nothing to refund. YAY!')
else:
    print(f'Noul value {noul} is in the uncertainty interval [{NEGATIVE_NOUL_THRESHOLD}-{POSITIVE_NOUL_THRESHOLD}]')
    print('Cannot take a confident decision.')

Refunding request detected. Sigh...


## Choice

### Una `choice` è una domanda `N`-aria: la risposta può essere una delle `N` etichette testuali ("criteri") che indichiamo al modello.

In [18]:
TEAM_CHOICE_NAME = 'team'

In [19]:
TEAM_CHOICE_BILLING_CRITERIA = 'billing'
TEAM_CHOICE_TECHNICAL_CRITERIA = 'technical'
TEAM_CHOICE_OTHER_CRITERIA = 'other'

### Per le `choice` è ovviamente obbligatorio elencare i possibili criteri, però le loro corrispondenti descrizioni rimangono facoltative.

In [20]:
TEAM_CHOICE = {
    TEAM_CHOICE_NAME: {
        'type': 'choice',
        'instructions': 'Which team should handle this ticket?',
        'criteria': {
            TEAM_CHOICE_BILLING_CRITERIA: 'Payments and refunds',
            TEAM_CHOICE_TECHNICAL_CRITERIA: 'Bugs and integrations',
            TEAM_CHOICE_OTHER_CRITERIA: 'None of the above',
        },
    }
}

In [21]:
print_dictionary(TEAM_CHOICE)

{
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this ticket?",
        "criteria": {
            "billing": "Payments and refunds",
            "technical": "Bugs and integrations",
            "other": "None of the above"
        }
    }
}


In [22]:
response = ollama.systemone(DECISION_MODEL_NAME, TICKET, TEAM_CHOICE)

### Per le choice esiste un punteggio di confidenza a parte. Anche in questo caso, può essere conveniente sfruttarlo per individuare risposte troppo "incerte". 

In [23]:
MIN_CONFIDENCE = 0.6

In [24]:
answer = response.answers[TEAM_CHOICE_NAME]

confidence = answer.confidence

if answer.confidence >= MIN_CONFIDENCE:
    if (answer.choice == TEAM_CHOICE_BILLING_CRITERIA):
        print('Submitting the ticket to the billing team.')
    elif (answer.choice == TEAM_CHOICE_TECHNICAL_CRITERIA):
        print('Submitting the ticket to the technical team.')
    elif (answer.choice == TEAM_CHOICE_OTHER_CRITERIA):
        print('Submitting the ticket to the general team')
    else:
        raise BaseException(f'An unexpected choice for the "{TEAM_CHOICE_NAME}" question was detected.')
else:
    print(f'Confidence ({confidence}) is lower than the threshold ({MIN_CONFIDENCE}).')
    print('Cannot determine what team should handle this ticket.')

Submitting the ticket to the billing team.


## Score

### Lo `score` è una domanda la cui risposta è un valore numerico compreso tra `1.0` e `N - 1`, dove `N` è la quantità di possibili criteri che indichiamo al modello.

In [25]:
URGENCY_SCORE_NAME = 'urgency'

### Anche in questo caso è ovviamente obbligatorio elencare tutti i criteri.

In [26]:
URGENCY_SCORE_CRITERIAS = ['Routine', 'Soon', 'Urgent']

In [27]:
URGENCY_MIN_SCORE = 0.0
URGENCY_MAX_SCORE = float(len(URGENCY_SCORE_CRITERIAS) - 1)

In [28]:
URGENCY_SCORE = {
    URGENCY_SCORE_NAME: {
        'type': 'score',
        'instructions': 'How urgent is this ticket?',
        'criteria': URGENCY_SCORE_CRITERIAS
    }
}

In [29]:
print_dictionary(URGENCY_SCORE)

{
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this ticket?",
        "criteria": [
            "Routine",
            "Soon",
            "Urgent"
        ]
    }
}


In [30]:
response = ollama.systemone(DECISION_MODEL_NAME, TICKET, URGENCY_SCORE)

### Possiamo per esempio sfruttare questo punteggio per inserire i ticket di supporto in una coda ordinata per urgenza 

In [31]:
tickets_queue = [
    {
        'urgency': URGENCY_MAX_SCORE,
        'ticket': 'My computer is literally on fire pls HELPPPP 😭😭😭'
    },
    {
        'urgency': URGENCY_MIN_SCORE,
        'ticket': 'i dunno there\'s a small glitch on the website i guess...'
    },
]

In [32]:
answer = response.answers[URGENCY_SCORE_NAME]

if answer.confidence >= MIN_CONFIDENCE:
    tickets_queue.append({
        'urgency': answer.score,
        'ticket': TICKET,
    })

    tickets_queue.sort(key=lambda ticket: ticket[URGENCY_SCORE_NAME], reverse=True)
else:
    print(f'Confidence is lower than the threshold ({MIN_CONFIDENCE}).')
    print('Cannot determine the urgency of this ticket.')

In [33]:
for i, ticket in enumerate(tickets_queue, start=1):
    print(f'#{i} most urgent ticket')
    print_dictionary(ticket)
    print()

#1 most urgent ticket
{
    "urgency": 2.0,
    "ticket": "My computer is literally on fire pls HELPPPP \ud83d\ude2d\ud83d\ude2d\ud83d\ude2d"
}

#2 most urgent ticket
{
    "urgency": 1.8803322356071965,
    "ticket": "I was charged twice. Refund the extra payment immediately!"
}

#3 most urgent ticket
{
    "urgency": 0.0,
    "ticket": "i dunno there's a small glitch on the website i guess..."
}



## Parallel Questions

### Il modello è in grado di rispondere a più domande contemporaneamente, senza che questo impatti granché sulle performance.

In [34]:
TICKET_QUESTIONS = (REFUND_NOUL | TEAM_CHOICE | URGENCY_SCORE)

In [35]:
print_dictionary(TICKET_QUESTIONS)

{
    "refund": {
        "type": "noul",
        "instructions": "Does the customer explicitly ask for a refund?",
        "criteria": {
            "true": "The customer asks to be paid back because they are disattisfied or an error occured.",
            "false": "The customer may refer to money, but not asking to be paid back."
        }
    },
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this ticket?",
        "criteria": {
            "billing": "Payments and refunds",
            "technical": "Bugs and integrations",
            "other": "None of the above"
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this ticket?",
        "criteria": [
            "Routine",
            "Soon",
            "Urgent"
        ]
    }
}


In [36]:
response = ollama.systemone(DECISION_MODEL_NAME, TICKET, TICKET_QUESTIONS)

### Possiamo sfruttare tutti questi output per gestire automaticamente uno stream di ticket.

### Per ciascun messaggio, determiniamo il team che se ne deve occupare.

### Se questo il team della contabilità, stabiliamo se si tratta di una richiesta di rimborso, altrimenti ignoriamo la risposta a quella domanda. 

### Dopodiché inseriamo il ticket nella coda del team corretto, tenendola ordinata per urgenza.

### Se una qualsiasi di queste decisioni risulta essere troppo incerta, il ticket viene messo in un buffer di revisione manuale.

In [37]:
def ask_ticket_questions(ticket):
    response = ollama.systemone(DECISION_MODEL_NAME, ticket, TICKET_QUESTIONS)
    return response.answers

In [38]:
class LowConfidenceException(BaseException):
    confidence: float
    question: str

    def __init__(self, question, confidence):
        super().__init__(f'Confidence {confidence} for question "{question}" is loo low.')
        self.confidence = confidence
        self.question = question

In [39]:
def get_team(answers):
    answer = answers[TEAM_CHOICE_NAME]
    confidence = answer.confidence

    if confidence < MIN_CONFIDENCE:
        raise LowConfidenceException(TEAM_CHOICE_NAME, confidence)

    return answer.choice

In [40]:
def get_urgency(answers):
    answer = answers[URGENCY_SCORE_NAME]
    confidence = answer.confidence

    if confidence < MIN_CONFIDENCE:
        raise LowConfidenceException(URGENCY_SCORE_NAME, confidence)

    return answer.score

In [41]:
def is_refund_request(answers):
    noul = answers[REFUND_NOUL_NAME].noul

    if noul > POSITIVE_NOUL_THRESHOLD:
        return True
    elif noul < NEGATIVE_NOUL_THRESHOLD:
        return False
    else:
        raise LowConfidenceException(REFUND_NOUL_NAME, noul)

In [42]:
def add_to_tickets_queue(tickets_queue, ticket):
    tickets_queue.append(ticket)
    tickets_queue.sort(key=lambda ticket: ticket['urgency'], reverse=True)

In [43]:
TICKETS_STREAM = [
    "I need a refund for my subscription, I was accidentally charged twice this month.",
    "How do I reset my password? The forgot password link isn't sending an email to my inbox.",
    "The application crashes immediately upon startup after the latest update on macOS.",
    "Where can I update my billing address and credit card information?",
    "My account has been locked and I can't access my team's dashboard.",
    "Can I get a 7-day extension on my trial period? I haven't had time to test the enterprise features.",
    "API requests are returning a 500 internal server error for the past hour.",
    "I want to cancel my account and delete all my data permanently from your servers.",
    "The export to CSV feature is missing data from the custom fields column.",
    "How do I add another team member to my workspace and manage their role permissions?"
]

In [44]:
review_ticket_buffer = []

team_tickets_queues = {
    TEAM_CHOICE_BILLING_CRITERIA: [],
    TEAM_CHOICE_TECHNICAL_CRITERIA: [],
    TEAM_CHOICE_OTHER_CRITERIA: []
}

for ticket_msg in tqdm(TICKETS_STREAM):
    try:
        answers = ask_ticket_questions(ticket_msg)
    
        team = get_team(answers)
        urgency = get_urgency(answers)

        ticket = {
            'message': ticket_msg,
            'urgency': urgency
        }
        
        if team == TEAM_CHOICE_BILLING_CRITERIA:
            ticket['refund_request'] = is_refund_request(answers)

        team_tickets_queue = team_tickets_queues[team]
        add_to_tickets_queue(team_tickets_queue, ticket)
    except LowConfidenceException as e:
        ticket = {
            'message': ticket_msg,
            'problem': {
                'question': e.question,
                'confidence': e.confidence
            }
        }
        
        review_ticket_buffer.append(ticket)

  0%|          | 0/10 [00:00<?, ?it/s]

In [45]:
for team, queue in team_tickets_queues.items():
    print('=================')
    print(f'Team: {team}')
    print('=================')

    for i, ticket in enumerate(queue, start=1):
        print(f'#{i} most urgent ticket')
        print_dictionary(ticket)
    
    print()

Team: billing
#1 most urgent ticket
{
    "message": "Where can I update my billing address and credit card information?",
    "urgency": 0.04678059487639691,
    "refund_request": false
}

Team: technical
#1 most urgent ticket
{
    "message": "API requests are returning a 500 internal server error for the past hour.",
    "urgency": 1.920828131835988
}
#2 most urgent ticket
{
    "message": "The application crashes immediately upon startup after the latest update on macOS.",
    "urgency": 1.9022134777233723
}
#3 most urgent ticket
{
    "message": "The export to CSV feature is missing data from the custom fields column.",
    "urgency": 0.16595546312410753
}

Team: other
#1 most urgent ticket
{
    "message": "How do I reset my password? The forgot password link isn't sending an email to my inbox.",
    "urgency": 0.2336194152053141
}
#2 most urgent ticket
{
    "message": "I want to cancel my account and delete all my data permanently from your servers.",
    "urgency": 0.201837796

In [46]:
print('=================')
print("Uncertain tickets")
print('=================')

for ticket_msg in review_ticket_buffer:
    print_dictionary(ticket_msg)

Uncertain tickets
{
    "message": "I need a refund for my subscription, I was accidentally charged twice this month.",
    "problem": {
        "question": "urgency",
        "confidence": 0.0
    }
}
{
    "message": "My account has been locked and I can't access my team's dashboard.",
    "problem": {
        "question": "urgency",
        "confidence": 0.0
    }
}


In [47]:
ollama_process.kill()